### Importing all required libraries

In [1]:
# !pip uninstall flwr -y
# !pip install flwr==1.7.0

In [2]:
import os
import glob
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Dense, BatchNormalization, Dropout, Concatenate,
    GlobalAveragePooling2D, Conv2D, MaxPooling2D
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.utils import Sequence
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

import flwr as fl

warnings.filterwarnings("ignore")

print("All libraries imported successfully")
print("Flower version:", fl.__version__)
print("TensorFlow version:", tf.__version__)

All libraries imported successfully
Flower version: 1.7.0
TensorFlow version: 2.19.0


### Kaggle setup

In [3]:
if not os.path.exists('/root/.kaggle'):
    os.makedirs('/root/.kaggle')

!mv kaggle.json /root/.kaggle/ 2>/dev/null || true
!chmod 600 /root/.kaggle/kaggle.json 2>/dev/null || true

### Extracting dataset from kaggle

In [4]:
!kaggle datasets download -d mehradaria/leukemia -p /content/

with zipfile.ZipFile('/content/leukemia.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/A_Raw/')

Dataset URL: https://www.kaggle.com/datasets/mehradaria/leukemia
License(s): ODbL-1.0
100% 110M/110M [00:07<00:00, 15.3MB/s]



### Structuring folder

In [5]:
base_path = '/content/Institution_Alpha'
os.makedirs(f'{base_path}/Healthy',   exist_ok=True)
os.makedirs(f'{base_path}/Leukemia',  exist_ok=True)

In [6]:
for p in glob.glob('/content/A_Raw/**/Benign/*.*', recursive=True):
    shutil.copy(p, f'{base_path}/Healthy/')

for sub in ['Early', 'Pre', 'Pro']:
    for p in glob.glob(f'/content/A_Raw/**/{sub}/*.*', recursive=True):
        shutil.copy(p, f'{base_path}/Leukemia/')

print(f"Institution Alpha Ready: {len(os.listdir(base_path+'/Healthy'))} Healthy, " f"{len(os.listdir(base_path+'/Leukemia'))} Leukemia images.")


Institution Alpha Ready: 504 Healthy, 2752 Leukemia images.


### Importing Clinical Data

In [7]:
df_a = pd.read_csv('/content/hospital_A_clinical.csv')

In [8]:
def remap_path(old_path):
    filename = os.path.basename(old_path)

    for prefix in ['A_Pre_', 'A_Early_', 'A_Pro_', 'A_Benign_', 'A_H_', 'A_L_']:
        if filename.startswith(prefix):
            filename = filename.replace(prefix, '')
            break

    subfolder = 'Healthy' if ('Healthy' in old_path or '_H_' in filename or 'Benign' in old_path) else 'Leukemia'

    return f'/content/Institution_Alpha/{subfolder}/{filename}'

In [9]:
df_a['image_path'] = df_a['image_path'].apply(remap_path)

KeyError: 'image_path'

In [ ]:
valid_paths = df_a['image_path'].apply(os.path.exists)
missing_count = (~valid_paths).sum()

if missing_count > 0:
    print(f"Warning: {missing_count} images not found on disk. Dropping missing rows...")
    df_a = df_a[valid_paths].reset_index(drop=True)
else:
    print("All CSV image paths successfully verified.")

print(f"Total valid patients locked in for training: {len(df_a)}")

In [ ]:
features = ['WBC_count', 'LDH_level', 'Hemoglobin', 'Platelet_count',
            'RBC_count', 'Hematocrit', 'Lymphocyte_percentage', 'Neutrophil_percentage', 'Uric_acid']

In [ ]:
unique_patients = df_a['Patient_ID'].unique()
print("Total patients:", len(unique_patients))

### Spliting data into train and test

In [ ]:
train_ids, temp_ids = train_test_split(
    unique_patients, test_size=0.3, random_state=42
)

val_ids, test_ids = train_test_split(
    temp_ids, test_size=0.5, random_state=42
)

train_df = df_a[df_a['Patient_ID'].isin(train_ids)].reset_index(drop=True)
val_df   = df_a[df_a['Patient_ID'].isin(val_ids)].reset_index(drop=True)
test_df  = df_a[df_a['Patient_ID'].isin(test_ids)].reset_index(drop=True)

In [ ]:
print("Train-Test overlap:",
      len(set(train_df['Patient_ID']) & set(test_df['Patient_ID'])))

print("Train-Val overlap:",
      len(set(train_df['Patient_ID']) & set(val_df['Patient_ID'])))

print("Val-Test overlap:",
      len(set(val_df['Patient_ID']) & set(test_df['Patient_ID'])))

### Scaling data

In [ ]:
scaler = StandardScaler()

train_df[features] = scaler.fit_transform(train_df[features])

val_df[features]   = scaler.transform(val_df[features])
test_df[features]  = scaler.transform(test_df[features])

print(f"split done: {len(train_df)} Train, {len(val_df)} Val, {len(test_df)} Test rows.")

### Data Agumentation

In [ ]:
aug = ImageDataGenerator(
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.85, 1.15],
    fill_mode='nearest'
)

### THE MULTIMODAL DATA ENGINE

In [ ]:
class LeukemiaDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size=32, target_size=(224, 224), shuffle=True, is_training=True, mixup_alpha=0.2):
        self.df = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.target_size = target_size
        self.shuffle = shuffle
        self.is_training = is_training
        self.mixup_alpha = mixup_alpha
        self.features = ['WBC_count', 'LDH_level', 'Hemoglobin', 'Platelet_count',
                         'RBC_count', 'Hematocrit', 'Lymphocyte_percentage', 'Neutrophil_percentage', 'Uric_acid']
        self.on_epoch_end()

    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac=1).reset_index(drop=True)

    def __getitem__(self, index):
        batch = self.df.iloc[index * self.batch_size : (index + 1) * self.batch_size]

        images = []
        for p in batch['image_path']:
            img = img_to_array(load_img(p, target_size=self.target_size)) / 255.0
            if self.is_training:
                img = aug.random_transform(img)
            images.append(img)
        X_img = np.array(images)

        X_tab = batch[self.features].values.astype(np.float32)
        y = batch['Diagnosis'].values.astype(np.float32)

        return (X_img, X_tab), y

In [ ]:
train_gen = LeukemiaDataGenerator(train_df, batch_size=32, is_training=True,  mixup_alpha=0.2)
val_gen   = LeukemiaDataGenerator(val_df,   batch_size=32, is_training=False, shuffle=False)
test_gen  = LeukemiaDataGenerator(test_df,  batch_size=32, is_training=False, shuffle=False)

### THE DUAL-INPUT ARCHITECTURE

In [ ]:
def build_multimodal_model():

    # --- IMAGE BRANCH (Custom CNN — no pretrained weights) ---
    img_input = Input(shape=(224, 224, 3), name="image_input")

    # Block 1
    x = Conv2D(16, (3, 3), activation='relu', padding='same',kernel_regularizer=l2(1e-4))(img_input)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.2)(x)

    # Block 2
    x = Conv2D(32, (3, 3), activation='relu', padding='same',kernel_regularizer=l2(1e-4))(x)
    x = MaxPooling2D((2, 2))(x)
    x = Dropout(0.25)(x)

    x = GlobalAveragePooling2D()(x)
    img_features = Dense(32, activation='relu',kernel_regularizer=l2(1e-4))(x)

    # --- CLINICAL BRANCH (MLP) ---
    tab_input = Input(shape=(10,), name="clinical_input")
    t = Dense(64, activation='relu',kernel_regularizer=l2(1e-4))(tab_input)
    t = BatchNormalization()(t)
    t = Dropout(0.3)(t)
    t = Dense(32, activation='relu',kernel_regularizer=l2(1e-4))(t)
    tab_features = Dropout(0.2)(t)

    # --- FUSION LAYER ---
    merged = Concatenate()([img_features, tab_features])
    z = Dense(64, activation='relu', kernel_regularizer=l2(1e-4))(merged)
    z = Dropout(0.35)(z)

    output   = Dense(1, activation='sigmoid', name="diagnosis")(z)

    model = Model(inputs=[img_input, tab_input], outputs=output)

    model.compile(
        optimizer=Adam(0.0001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
local_model = build_multimodal_model()
local_model.summary()

### THE PRIVACY WRAPPER (FLOWER CLIENT)

In [ ]:
class LeukemiaClient(fl.client.NumPyClient):
    def __init__(self, model, train_generator, val_generator):
        self.model     = model
        self.train_gen = train_generator
        self.val_gen   = val_generator

    def get_parameters(self, config):
        """Differential Privacy: add tiny Gaussian noise before sharing weights."""
        weights = self.model.get_weights()
        noisy_weights = [w + np.random.normal(0, 0.001, w.shape) for w in weights]
        return noisy_weights

    def fit(self, parameters, config):
        """Receive global weights, train locally, return updated weights."""
        self.model.set_weights(parameters)
        print("\nTraining on local patient silo...")
        self.model.fit(self.train_gen, epochs=1, verbose=1)
        return self.get_parameters(config={}), len(self.train_gen.df), {}

    def evaluate(self, parameters, config):
        """Evaluate global model on local validation set."""
        self.model.set_weights(parameters)
        print("\nEvaluating Global Model on local held-out validation set...")
        loss, accuracy = self.model.evaluate(self.val_gen, verbose=0)
        return float(loss), len(self.val_gen.df), {"accuracy": float(accuracy)}

print("Federated Client Logic Initialized.")

### Training

In [ ]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model_alpha.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
history = local_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'],     label='Train Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='orange')
plt.title('Multimodal Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'],     label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='orange')
plt.title('Multimodal Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

print("Local architecture verified! The Data Generator and Model are perfectly synced.")

In [ ]:
def evaluate_on_test(model, df):
    patient_preds = []
    patient_true  = []

    for patient in df['Patient_ID'].unique():
        patient_data = df[df['Patient_ID'] == patient]

        preds = []
        labels = []

        for _, row in patient_data.iterrows():
            img = img_to_array(load_img(row['image_path'], target_size=(224,224))) / 255.0
            tab = row[features].values.astype(np.float32)

            pred = model.predict(
                [np.expand_dims(img,0), np.expand_dims(tab,0)],
                verbose=0
            )

            preds.append(pred[0][0])
            labels.append(row['Diagnosis'])

        final_pred = int(np.mean(preds) > 0.5)
        true_label = int(np.mean(labels) > 0.5)

        patient_preds.append(final_pred)
        patient_true.append(true_label)

    print("\n--- PATIENT LEVEL RESULTS ---")
    print(classification_report(patient_true, patient_preds))

In [ ]:
evaluate_on_test(local_model, test_df)

### Connecting to Global Model

In [ ]:
SERVER_URL = "PASTE_YOUR_PUBLIC_URL_HERE"

CLEAN_URL = SERVER_URL.replace("https://", "").strip("/")

print(f"Dialing Global Server at: {CLEAN_URL}...")

fl.client.start_numpy_client(
    server_address=CLEAN_URL,
    client=LeukemiaClient(local_model, train_gen, val_gen)
)